In [59]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MultiLabelBinarizer
import optuna

# EDA

In [37]:
df = pd.read_csv('./steam.csv')
df.head()

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,124534,3339,17612,317,10000000-20000000,7.19
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,3318,633,277,62,5000000-10000000,3.99
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Valve Anti-Cheat enabled,Action,FPS;World War II;Multiplayer,0,3416,398,187,34,5000000-10000000,3.99
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,Multi-player;Online Multi-Player;Local Multi-P...,Action,Action;FPS;Multiplayer,0,1273,267,258,184,5000000-10000000,3.99
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,Single-player;Multi-player;Valve Anti-Cheat en...,Action,FPS;Action;Sci-fi,0,5250,288,624,415,5000000-10000000,3.99


In [38]:
df.shape

(27075, 18)

In [39]:
df.isnull().sum()

appid                0
name                 0
release_date         0
english              0
developer            1
publisher           14
platforms            0
required_age         0
categories           0
genres               0
steamspy_tags        0
achievements         0
positive_ratings     0
negative_ratings     0
average_playtime     0
median_playtime      0
owners               0
price                0
dtype: int64

In [40]:
df[df.isna().any(axis=1)]

,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,steamspy_tags,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price
3420,307170,Borealis,2014-09-02,1,Conrad Nelson,NaN,windows,0,Single-player;Steam Achievements;Steam Trading...,Action;Casual;Indie,Casual;Indie;Action,17,301,192,132,146,50000-100000,3.99
4511,341120,Glorkian Warrior: The Trials Of Glork,2015-03-24,1,Pixeljam,NaN,windows;mac,0,Single-player;Steam Achievements;Full controll...,Indie,Indie;Platformer;Shoot 'Em Up,18,234,26,274,274,20000-50000,2.79
5229,359370,Pirate's Life,2015-04-17,1,Team Eyepatch,NaN,windows,0,Single-player,Simulation;Strategy,Strategy;Simulation;Pirates,0,25,41,0,0,0-20000,3.99
7464,422940,Divergence: Online,2016-01-06,1,Stained Glass Llama,NaN,windows,0,Multi-player;MMO;Steam Turn Notifications,Action;Indie;Massively Multiplayer;RPG;Early A...,Early Access;Massively Multiplayer;Indie,0,72,51,0,0,0-20000,14.99
7911,436240,Melancholy Republic,2018-04-12,1,Cloud Runner Studios,NaN,windows,0,Single-player,Adventure;Casual;Indie,Adventure;Indie;Casual,0,14,6,0,0,0-20000,6.99
9894,498710,After Dreams,2018-05-06,1,Matt Boyer,NaN,windows,0,Single-player,Violent;Adventure;Casual;Free to Play;Indie;Si...,Free to Play;Adventure;Indie,0,126,47,1,1,20000-50000,0.00
10011,502150,Interstellar Logistics Inc,2016-08-15,1,Exalted Guy Interactive,NaN,windows,0,Single-player,Casual,Casual;Puzzle;Sci-fi,9,25,4,281,281,0-20000,2.09
10564,516430,Ruin of the Reckless,2017-04-26,1,Faux-Operative Games,NaN,windows,0,Single-player;Co-op;Local Co-op;Shared/Split S...,Action;Adventure;Indie;RPG,Action;Indie;Adventure,0,63,11,0,0,0-20000,6.99
11123,531240,Max Stern,2016-10-21,1,Lupan Artiom Oleg,NaN,windows;mac;linux,0,Single-player;Steam Achievements;Steam Trading...,Action;Adventure;Indie,Action;Indie;Adventure,10,17,13,266,266,0-20000,3.99
13901,610740,SuperCluster: Void,2017-05-15,1,Logan McClure,NaN,windows,0,Single-player,Adventure;Indie;RPG,Adventure;RPG;Indie,0,16,4,0,0,0-20000,3.99


In [41]:
def encode_multilabel(df, col):
    mlb = MultiLabelBinarizer()
    split = df[col].fillna("").apply(
        lambda x: [i.strip() for i in x.split(";") if i.strip()]
    )

    encoded = mlb.fit_transform(split)

    encoded_df = pd.DataFrame(
        encoded,
        columns=mlb.classes_,
        index=df.index  # 🔥 важно
    )

    df = pd.concat([df, encoded_df], axis=1)
    df = df.drop(columns=[col])

    return df

In [42]:
df = encode_multilabel(df, 'categories')
df = encode_multilabel(df, 'genres')
df = encode_multilabel(df, 'steamspy_tags')
df

,appid,name,release_date,english,developer,publisher,platforms,required_age,achievements,positive_ratings,...,Warhammer 40K,Web Publishing,Werewolves,Western,Word Game,World War I,World War II,Wrestling,Zombies,e-sports
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,0,124534,...,0,0,0,0,0,0,0,0,0,0
1,20,Team Fortress Classic,1999-04-01,1,Valve,Valve,windows;mac;linux,0,0,3318,...,0,0,0,0,0,0,0,0,0,0
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,0,3416,...,0,0,0,0,0,0,1,0,0,0
3,40,Deathmatch Classic,2001-06-01,1,Valve,Valve,windows;mac;linux,0,0,1273,...,0,0,0,0,0,0,0,0,0,0
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,0,5250,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27070,1065230,Room of Pandora,2019-04-24,1,SHEN JIAWEI,SHEN JIAWEI,windows,0,7,3,...,0,0,0,0,0,0,0,0,0,0
27071,1065570,Cyber Gun,2019-04-23,1,Semyon Maximov,BekkerDev Studio,windows,0,0,8,...,0,0,0,0,0,0,0,0,0,0
27072,1065650,Super Star Blast,2019-04-24,1,EntwicklerX,EntwicklerX,windows,0,24,0,...,0,0,0,0,0,0,0,0,0,0
27073,1066700,New Yankee 7: Deer Hunters,2019-04-17,1,Yustas Game Studio,Alawar Entertainment,windows;mac,0,0,2,...,0,0,0,0,0,0,0,0,0,0


In [43]:
df.at[23071, 'developer'] = 'None'

In [44]:
df['publisher'] = df['publisher'].fillna(df['developer'])

In [45]:
df.isnull().sum()

appid           0
name            0
release_date    0
english         0
developer       0
               ..
World War I     0
World War II    0
Wrestling       0
Zombies         0
e-sports        0
Length: 412, dtype: int64

In [46]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 27075 entries, 0 to 27074
Columns: 412 entries, appid to e-sports
dtypes: float64(1), int64(405), str(6)
memory usage: 85.1 MB


In [47]:
df.describe()

,appid,english,required_age,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,price,Captions available,...,Warhammer 40K,Web Publishing,Werewolves,Western,Word Game,World War I,World War II,Wrestling,Zombies,e-sports
count,2.707500e+04,27075.000000,27075.000000,27075.000000,2.707500e+04,27075.000000,27075.000000,27075.00000,27075.000000,27075.000000,...,27075.000000,27075.000000,27075.000000,27075.000000,27075.000000,27075.000000,27075.000000,27075.000000,27075.000000,27075.000000
mean,5.962035e+05,0.981127,0.354903,45.248864,1.000559e+03,211.027147,149.804949,146.05603,6.078193,0.026630,...,0.001071,0.000332,0.000111,0.000849,0.000369,0.000849,0.004801,0.000295,0.005836,0.000074
std,2.508942e+05,0.136081,2.406044,352.670281,1.898872e+04,4284.938531,1827.038141,2353.88008,7.874922,0.161002,...,0.032711,0.018229,0.010526,0.029134,0.019215,0.029134,0.069127,0.017187,0.076170,0.008595
min,1.000000e+01,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.00000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,4.012300e+05,1.000000,0.000000,0.000000,6.000000e+00,2.000000,0.000000,0.00000,1.690000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5.990700e+05,1.000000,0.000000,7.000000,2.400000e+01,9.000000,0.000000,0.00000,3.990000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,7.987600e+05,1.000000,0.000000,23.000000,1.260000e+02,42.000000,0.000000,0.00000,7.190000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.069460e+06,1.000000,18.000000,9821.000000,2.644404e+06,487076.000000,190625.000000,190625.00000,421.990000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [48]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')

KeyboardInterrupt: 

# Modeling

In [49]:
X = df[['price', 'achievements', 'positive_ratings','negative_ratings']]
y = df['average_playtime']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [50]:
lr = LinearRegression()

lr.fit(x_train, y_train)

y_train_pred = lr.predict(x_train)
y_test_pred = lr.predict(x_test)

In [51]:
def grade_model(grade):
    print(f'Train {grade.__name__}: {grade(y_train, y_train_pred):.4f}')
    print(f'Test {grade.__name__}: {grade(y_test, y_test_pred):.4f}')
    print('-'*25)

In [52]:
grade_model(mean_squared_error)
grade_model(mean_absolute_error)
grade_model(r2_score)

Train mean_squared_error: 1616657.2369
Test mean_squared_error: 7019627.3297
-------------------------
Train mean_absolute_error: 201.5035
Test mean_absolute_error: 242.8856
-------------------------
Train r2_score: 0.0587
Test r2_score: 0.0137
-------------------------


In [53]:
#попробуем обучить случайный лес

rfc = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42
)

rfc.fit(x_train, y_train)

y_train_pred = rfc.predict(x_train)
y_test_pred = rfc.predict(x_test)

grade_model(mean_squared_error)
grade_model(mean_absolute_error)
grade_model(r2_score)

Train mean_squared_error: 1191250.1048
Test mean_squared_error: 6967801.7276
-------------------------
Train mean_absolute_error: 143.5972
Test mean_absolute_error: 207.0357
-------------------------
Train r2_score: 0.3064
Test r2_score: 0.0210
-------------------------


In [58]:

def objective(trial):
    # Подбираем параметры так, чтобы снизить переобучение
    params = {
        # Количество деревьев (больше — стабильнее, но медленнее)
        "n_estimators": trial.suggest_int("n_estimators", 50, 300),
        # Глубина: ограничиваем сверху, чтобы модель не зазубривала данные
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        # Минумум объектов в листе: чем больше число, тем сильнее регуляризация
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 50),
        # Минумум объектов для разделения узла
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 100),
        # Количество признаков для расщепления
        "max_features": trial.suggest_float("max_features", 0.5, 1.0),
        "random_state": 42,
        "n_jobs": -1,  # использовать все ядра процессора для ускорения
    }

    # Обучаем модель
    model = RandomForestRegressor(**params)
    model.fit(x_train, y_train)

    # Делаем предсказание на ТЕСТОВЫХ данных
    y_test_pred = model.predict(x_test)

    # Оптимизируем R2-score на тесте (цель — поднять его выше 0.02)
    score = r2_score(y_test, y_test_pred)

    return score


# Запуск поиска
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)  # начните с 30 итераций

print("Лучший R2 на тесте:", study.best_value)
print("Лучшие параметры:", study.best_params)

[I 2026-07-03 10:44:58,927] A new study created in memory with name: no-name-e2b235cd-7b26-4648-a033-9daede757c11
[I 2026-07-03 10:44:59,323] Trial 0 finished with value: 0.018872170329739535 and parameters: {'n_estimators': 57, 'max_depth': 10, 'min_samples_leaf': 33, 'min_samples_split': 56, 'max_features': 0.5655994969325175}. Best is trial 0 with value: 0.018872170329739535.
[I 2026-07-03 10:44:59,565] Trial 1 finished with value: 0.016789677784403767 and parameters: {'n_estimators': 77, 'max_depth': 3, 'min_samples_leaf': 29, 'min_samples_split': 62, 'max_features': 0.5336768123201354}. Best is trial 0 with value: 0.018872170329739535.
[I 2026-07-03 10:44:59,859] Trial 2 finished with value: 0.02132800287530945 and parameters: {'n_estimators': 51, 'max_depth': 9, 'min_samples_leaf': 9, 'min_samples_split': 41, 'max_features': 0.6271327207474717}. Best is trial 2 with value: 0.02132800287530945.
[I 2026-07-03 10:45:00,716] Trial 3 finished with value: 0.015023299513867494 and param

Лучший R2 на тесте: 0.023035032006104794
Лучшие параметры: {'n_estimators': 228, 'max_depth': 14, 'min_samples_leaf': 8, 'min_samples_split': 10, 'max_features': 0.8863824023297554}
